In [1]:
import pandas as pd
import numpy as np
import re, os, pickle, json
from pathlib import Path
from collections import Counter

In [2]:
# v2 splits   →  template-stripped captions from clean_data_part3
train_df = pd.read_csv('train_caption.csv')
val_df   = pd.read_csv('val_caption.csv')
test_df  = pd.read_csv('test_caption.csv')

print('train :', len(train_df))
print('val   :', len(val_df))
print('test  :', len(test_df))

train : 10984
val   : 1373
test  : 1374


In [3]:
train_df.head()

,local_path,description,room_type,wc
0,images/home-office/home-office_00740.jpg,freestanding desk carpeted and beige floor stu...,home-office,11
1,images/pool/poolroom_02998.jpg,Hot tub - backyard tile and l-shaped hot tub idea,pool,10
2,images/pool/poolroom_02439.jpg,Pool house - indoor tile and rectangular lap p...,pool,11
3,images/kitchen/kitchen_00862.jpg,1960s u-shaped light wood floor and brown floo...,kitchen,27
4,images/pool/poolroom_02841.jpg,Pool landscaping - backyard stone and rectangu...,pool,10


In [4]:
train_df.info()
print()
print(train_df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 10984 entries, 0 to 10983
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   local_path   10984 non-null  str  
 1   description  10984 non-null  str  
 2   room_type    10984 non-null  str  
 3   wc           10984 non-null  int64
dtypes: int64(1), str(3)
memory usage: 343.4 KB

local_path     0
description    0
room_type      0
wc             0
dtype: int64


# text cleaning   →  lowercase , remove numbers , special chars  (same as v1 stage 3)

In [5]:
def clean_caption_text(s):
    s = str(s).lower()
    s = re.sub(r'[0-9]+', '', s)
    s = re.sub(r"[^a-z\s']", ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


# example
ex = "Master carpeted bedroom with white walls and no fireplace, 1960s style"
print('BEFORE:', ex)
print('AFTER :', clean_caption_text(ex))

BEFORE: Master carpeted bedroom with white walls and no fireplace, 1960s style
AFTER : master carpeted bedroom with white walls and no fireplace s style


In [6]:
train_df['caption'] = train_df['description'].apply(clean_caption_text)
val_df['caption']   = val_df['description'].apply(clean_caption_text)
test_df['caption']  = test_df['description'].apply(clean_caption_text)

train_df[['description','caption']].head(3)

,description,caption
0,freestanding desk carpeted and beige floor stu...,freestanding desk carpeted and beige floor stu...
1,Hot tub - backyard tile and l-shaped hot tub idea,hot tub backyard tile and l shaped hot tub idea
2,Pool house - indoor tile and rectangular lap p...,pool house indoor tile and rectangular lap poo...


# build vocab from TRAIN only   (no leakage from val/test)
# notice this vocab will be SMALLER than v1   →   templates removed = fewer unique words

In [7]:
counter = Counter()
for cap in train_df['caption']:
    counter.update(cap.split())

print('total unique words in train v2 :', len(counter))
print('top 10 most common:', counter.most_common(10))

total unique words in train v2 : 11017
top 10 most common: [('and', 16405), ('a', 9949), ('with', 9804), ('floor', 9099), ('white', 6338), ('the', 5952), ('cabinets', 5710), ('wood', 5460), ('walls', 5421), ('tile', 4568)]


In [8]:
MIN_FREQ = 3

words = [w for w, c in counter.items() if c >= MIN_FREQ]
print(f'words with freq >= {MIN_FREQ}: {len(words)}')
print(f'words dropped (rare)         : {len(counter) - len(words)}')

words with freq >= 3: 4203
words dropped (rare)         : 6814


In [9]:
PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN = '<PAD>', '<START>', '<END>', '<UNK>'

word2idx = {
    PAD_TOKEN:   0,
    START_TOKEN: 1,
    END_TOKEN:   2,
    UNK_TOKEN:   3,
}

for w in sorted(words):
    word2idx[w] = len(word2idx)

idx2word   = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(word2idx)

print('vocab size :', VOCAB_SIZE)
print('first 10   :', list(word2idx.items())[:10])

vocab size : 4207
first 10   : [('<PAD>', 0), ('<START>', 1), ('<END>', 2), ('<UNK>', 3), ("'", 4), ("'s", 5), ("'x'", 6), ('a', 7), ('aaron', 8), ('ability', 9)]


# tokenize captions  →  text to integer IDs , pad to MAX_LEN
# format: [<START>, w1, w2, ..., <END>, <PAD>, <PAD>, ...]
#
# v2 captions are SHORTER  →  we lower MAX_CAPTION_LEN from 52 to 40  (saves compute)

In [10]:
MAX_CAPTION_LEN = 40   # shorter than v1 (was 52) since v2 captions are shorter


def tokenize(text):
    ids = [word2idx[START_TOKEN]]
    for w in text.split():
        ids.append(word2idx.get(w, word2idx[UNK_TOKEN]))
    ids.append(word2idx[END_TOKEN])

    if len(ids) < MAX_CAPTION_LEN:
        ids = ids + [word2idx[PAD_TOKEN]] * (MAX_CAPTION_LEN - len(ids))
    else:
        ids = ids[:MAX_CAPTION_LEN-1] + [word2idx[END_TOKEN]]

    return ids


# example
ex = train_df['caption'].iloc[0]
toks = tokenize(ex)
print('caption :', ex[:90])
print('tokens  :', toks[:15], '...')
print('length  :', len(toks))

caption : freestanding desk carpeted and beige floor study room with white walls
tokens  : [1, 1498, 1021, 585, 137, 345, 1440, 3607, 3160, 4147, 4120, 4048, 2, 0, 0] ...
length  : 40


In [11]:
train_df['tokens'] = train_df['caption'].apply(tokenize)
val_df['tokens']   = val_df['caption'].apply(tokenize)
test_df['tokens']  = test_df['caption'].apply(tokenize)

train_df[['caption','tokens']].head(2)

,caption,tokens
0,freestanding desk carpeted and beige floor stu...,"[1, 1498, 1021, 585, 137, 345, 1440, 3607, 316..."
1,hot tub backyard tile and l shaped hot tub idea,"[1, 1809, 3884, 265, 3782, 137, 2065, 3318, 18..."


In [12]:
# UNK rate sanity check
def unk_rate(tokens_list):
    total, unk = 0, 0
    for toks in tokens_list:
        for t in toks:
            if t == word2idx[PAD_TOKEN]:
                continue
            total += 1
            if t == word2idx[UNK_TOKEN]:
                unk += 1
    return unk / total if total else 0

print(f'train UNK rate : {unk_rate(train_df["tokens"]):.3%}')
print(f'val   UNK rate : {unk_rate(val_df["tokens"]):.3%}')
print(f'test  UNK rate : {unk_rate(test_df["tokens"]):.3%}')

train UNK rate : 2.509%
val   UNK rate : 3.131%
test  UNK rate : 3.469%


# image preprocessing config   (same as v1)
# the actual transform runs on-the-fly in the Dataset class in stage 4

In [13]:
IMG_SIZE = (224, 224)
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

print('IMG_SIZE :', IMG_SIZE)
print('IMG_MEAN :', IMG_MEAN)
print('IMG_STD  :', IMG_STD)

IMG_SIZE : (224, 224)
IMG_MEAN : [0.485, 0.456, 0.406]
IMG_STD  : [0.229, 0.224, 0.225]


# save outputs to a SEPARATE folder   →   preprocessing_outputs_v2/
# (don't overwrite the v1 stuff , in case we need to compare later)

In [14]:
OUT = Path('preprocessing_outputs')
OUT.mkdir(exist_ok=True)

vocab = {
    'word2idx':   word2idx,
    'idx2word':   idx2word,
    'vocab_size': VOCAB_SIZE,
    'pad_idx':    word2idx[PAD_TOKEN],
    'start_idx':  word2idx[START_TOKEN],
    'end_idx':    word2idx[END_TOKEN],
    'unk_idx':    word2idx[UNK_TOKEN],
}
with open(OUT / 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

train_df.to_csv(OUT / 'split_train.csv', index=False)
val_df.to_csv(OUT / 'split_val.csv',     index=False)
test_df.to_csv(OUT / 'split_test.csv',   index=False)

config = {
    'MAX_CAPTION_LEN': MAX_CAPTION_LEN,
    'MIN_FREQ':        MIN_FREQ,
    'VOCAB_SIZE':      VOCAB_SIZE,
    'IMG_SIZE':        list(IMG_SIZE),
    'IMG_MEAN':        IMG_MEAN,
    'IMG_STD':         IMG_STD,
    'PAD_IDX':   word2idx[PAD_TOKEN],
    'START_IDX': word2idx[START_TOKEN],
    'END_IDX':   word2idx[END_TOKEN],
    'UNK_IDX':   word2idx[UNK_TOKEN],
}
with open(OUT / 'preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

for f in sorted(OUT.iterdir()):
    print(f.name, '-', round(f.stat().st_size / 1024, 1), 'KB')

preprocessing_config.json - 0.3 KB
split_test.csv - 744.6 KB
split_train.csv - 5932.0 KB
split_val.csv - 741.3 KB
vocab.pkl - 82.8 KB


# summary

In [15]:
print(f'rows train     {len(train_df):,}')
print(f'rows val       {len(val_df):,}')
print(f'rows test      {len(test_df):,}')
print(f'vocab size     {VOCAB_SIZE}')
print(f'max caption    {MAX_CAPTION_LEN} words')
print()
print('next step  →  build model 1 / 2 / 3 notebooks pointing at preprocessing_outputs/')

rows train     10,984
rows val       1,373
rows test      1,374
vocab size     4207
max caption    40 words

next step  →  build model 1 / 2 / 3 notebooks pointing at preprocessing_outputs/
